In [1]:
!pip install transformers datasets scikit-learn pandas torch accelerate evaluate -q



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import evaluate

print("All libraries loaded! GPU Available:", torch.cuda.is_available())

C:\Users\athar\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries loaded! GPU Available: True


In [3]:
import pandas as pd
import numpy as np
import torch
import os
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

In [4]:
print("Loading dataset from Hugging Face...")

dataset = load_dataset("shahxeebhassan/human_vs_ai_sentences", split="train")
df = dataset.to_pandas()

df_sampled = df.sample(n=10000, random_state=42).reset_index(drop=True)

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_sampled['text'].tolist(), df_sampled['label'].tolist(), test_size=0.2, random_state=42
)

print(f"Dataset ready! Training samples: {len(train_texts)} | Testing samples: {len(test_texts)}")

Loading dataset from Hugging Face...
Dataset ready! Training samples: 8000 | Testing samples: 2000


In [5]:
print("--- Training Baseline Model (TF-IDF + Logistic Regression) ---")

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X_train_tfidf = vectorizer.fit_transform(train_texts)
X_test_tfidf = vectorizer.transform(test_texts)

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_tfidf, train_labels)

baseline_preds = baseline_model.predict(X_test_tfidf)
baseline_acc = accuracy_score(test_labels, baseline_preds)

print(f"Baseline Accuracy: {baseline_acc * 100:.2f}%\n")
print("Classification Report:\n", classification_report(test_labels, baseline_preds, target_names=["Human (0)", "AI (1)"]))

--- Training Baseline Model (TF-IDF + Logistic Regression) ---
Baseline Accuracy: 82.65%

Classification Report:
               precision    recall  f1-score   support

   Human (0)       0.83      0.81      0.82       974
      AI (1)       0.83      0.84      0.83      1026

    accuracy                           0.83      2000
   macro avg       0.83      0.83      0.83      2000
weighted avg       0.83      0.83      0.83      2000



In [6]:
print("Downloading RoBERTa Tokenizer...")
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Convert back to Hugging Face Dataset format
train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
test_dataset = Dataset.from_dict({"text": test_texts, "label": test_labels})

def tokenize_function(examples):
    # Truncate to 128 tokens to keep training fast, pad the rest
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing data...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

print("Tokenization complete!")

Tokenizing data...


Map: 100%|████████████████████████████████████████████████████████████████| 2000/2000 [00:00<00:00, 8583.23 examples/s]

Tokenization complete!


In [7]:
print("--- Fine-Tuning RoBERTa ---")

# Load the base model with a 2-class classification head
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Define accuracy metric
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Setup training arguments
training_args = TrainingArguments(
    output_dir="./roberta_ai_detector",
    learning_rate=2e-5,
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    num_train_epochs=2,             # 2 full passes over the data
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    fp16=True,                      # Turbocharge GPU training
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# Start training!
print("Starting training loop... (Check the progress bar!)")
trainer.train()


--- Fine-Tuning RoBERTa ---


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting training loop... (Check the progress bar!)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.242000,0.323771,0.885500
2,0.119600,0.366454,0.913000


TrainOutput(global_step=1000, training_loss=0.24517042922973634, metrics={'train_runtime': 302.0074, 'train_samples_per_second': 52.979, 'train_steps_per_second': 3.311, 'total_flos': 1052444221440000.0, 'train_loss': 0.24517042922973634, 'epoch': 2.0})

In [8]:
print("--- Final Results ---")

#RoBERTa's predictions on the test set (10000)
roberta_results = trainer.evaluate()
roberta_acc = roberta_results['eval_accuracy']

print(f"🥇 Baseline (TF-IDF + LogReg) Accuracy: {baseline_acc * 100:.2f}%")
print(f"🚀 Fine-Tuned (RoBERTa) Accuracy:       {roberta_acc * 100:.2f}%")

if roberta_acc > baseline_acc:
    print("\nConclusion: RoBERTa's deep semantic understanding successfully beat the statistical baseline!")
else:
    print("\nConclusion: The TF-IDF baseline held its own! (Increase the dataset size in Cell 1 to see RoBERTa pull ahead).")

--- Final Results ---


🥇 Baseline (TF-IDF + LogReg) Accuracy: 82.65%
🚀 Fine-Tuned (RoBERTa) Accuracy:       91.30%

Conclusion: RoBERTa's deep semantic understanding successfully beat the statistical baseline!


In [9]:

import torch
import torch.nn.functional as F

def test_custom_text(text):
    print(f"\n--- Analyzing Custom Text ---")
    print(f"Input: '{text}'\n")

    # 1. Test with TF-IDF Baseline
    text_tfidf = vectorizer.transform([text])
    baseline_pred = baseline_model.predict(text_tfidf)[0]
    baseline_prob = baseline_model.predict_proba(text_tfidf)[0]
    
    baseline_label = "AI (1)" if baseline_pred == 1 else "Human (0)"
    baseline_confidence = baseline_prob[baseline_pred] * 100
    
    print(f"📊 Baseline (TF-IDF) Prediction: {baseline_label} ({baseline_confidence:.2f}% confidence)")

    # 2. Test with Fine-Tuned RoBERTa
    # Tokenize the text and send it to the GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device) 
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    
    # Run the text through the deep learning model
    with torch.no_grad():
        outputs = model(**inputs)
        
    # Convert raw logits to percentages using Softmax
    probabilities = F.softmax(outputs.logits, dim=1).squeeze().tolist()
    
    # 0 = Human, 1 = AI
    human_prob = probabilities[0] * 100
    ai_prob = probabilities[1] * 100
    
    if ai_prob > human_prob:
        roberta_label = "AI (1)"
        roberta_confidence = ai_prob
    else:
        roberta_label = "Human (0)"
        roberta_confidence = human_prob
        
    print(f"🤖 RoBERTa Prediction:           {roberta_label} ({roberta_confidence:.2f}% confidence)")
    print("-" * 40)

test_custom_text("Pokemon, is my favrite anime ever, it is such a nice piece of medoia and I abosolutely love it!")
test_custom_text("I was thinking earlier about how much of the human experience is just… messy overhead. Like that specific feeling of walking into a room and completely forgetting why you’re there, or the way a certain song can make you feel nostalgic for a place you’ve never actually been to. It’s not efficient, and it doesn't make a whole lot of logical sense, but it’s definitely the vibe.")


--- Analyzing Custom Text ---
Input: 'Pokemon, is my favrite anime ever, it is such a nice piece of medoia and I abosolutely love it!'

📊 Baseline (TF-IDF) Prediction: Human (0) (58.47% confidence)
🤖 RoBERTa Prediction:           Human (0) (99.86% confidence)
----------------------------------------

--- Analyzing Custom Text ---
Input: 'I was thinking earlier about how much of the human experience is just… messy overhead. Like that specific feeling of walking into a room and completely forgetting why you’re there, or the way a certain song can make you feel nostalgic for a place you’ve never actually been to. It’s not efficient, and it doesn't make a whole lot of logical sense, but it’s definitely the vibe.'

📊 Baseline (TF-IDF) Prediction: Human (0) (55.59% confidence)
🤖 RoBERTa Prediction:           AI (1) (99.73% confidence)
----------------------------------------


In [10]:
# Cell 7: Advanced Performance Metrics
import numpy as np
from sklearn.metrics import classification_report

print("--- Detailed Performance Breakdown ---\n")

# 1. Get raw predictions from the RoBERTa model on the test set
print("Gathering RoBERTa predictions...")
predictions_output = trainer.predict(tokenized_test)
roberta_logits = predictions_output.predictions
roberta_true_labels = predictions_output.label_ids

# 2. Convert raw logits (confidence scores) into final class predictions (0 or 1)
roberta_preds = np.argmax(roberta_logits, axis=-1)

# 3. Generate the comprehensive report for RoBERTa
print("\n🤖 RoBERTa Classification Report:")
print(classification_report(roberta_true_labels, roberta_preds, target_names=["Human (0)", "AI (1)"]))

# 4. Print the Baseline report again for a side-by-side comparison
print("\n📊 Baseline (TF-IDF + LogReg) Classification Report:")
print(classification_report(test_labels, baseline_preds, target_names=["Human (0)", "AI (1)"]))

--- Detailed Performance Breakdown ---

Gathering RoBERTa predictions...

🤖 RoBERTa Classification Report:
              precision    recall  f1-score   support

   Human (0)       0.97      0.85      0.90       974
      AI (1)       0.87      0.97      0.92      1026

    accuracy                           0.91      2000
   macro avg       0.92      0.91      0.91      2000
weighted avg       0.92      0.91      0.91      2000


📊 Baseline (TF-IDF + LogReg) Classification Report:
              precision    recall  f1-score   support

   Human (0)       0.83      0.81      0.82       974
      AI (1)       0.83      0.84      0.83      1026

    accuracy                           0.83      2000
   macro avg       0.83      0.83      0.83      2000
weighted avg       0.83      0.83      0.83      2000



In [11]:
# Cell 10: Save your trained RoBERTa model
import os

save_directory = "./roberta_ai_detector_final"

print(f"Saving model and tokenizer to '{save_directory}'...")

# Save the tokenizer and the model to your hard drive
tokenizer.save_pretrained(save_directory)
model.save_pretrained(save_directory)

print("✅ Model successfully saved! You can now safely close this notebook without losing your work.")

Saving model and tokenizer to './roberta_ai_detector_final'...
✅ Model successfully saved! You can now safely close this notebook without losing your work.
